In [1]:
import requests
import pandas as pd

In [2]:
def get_user_id(name):
    num_results = 1
    try:
        page = requests.get(f'https://app.universaltennis.com/api/v2/search/players?query={name}&top={num_results}')
        data = page.json()
        # extract user id from either 'hits' or 'Hits' because utr has two JSON responses
        key = 'hits' if 'hits' in data else 'Hits'
        return data[key][0]['id'] if key == 'hits' else data[key][0]['Id']
    except (KeyError, IndexError, requests.exceptions.RequestException):
        print(f'Error: Could not find user ID for {name}')

In [ ]:
get_user_id('Roger Federer')

In [3]:
def get_utr(name):
    user_id = get_user_id(name)
    # get player utr
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}'
    page = requests.get(url)
    data = page.json()
    singles_utr = data['singlesUtr']
    return singles_utr

In [4]:
print(get_utr('Novak Djokovic'))
print(get_utr('Spencer Johnson'))

16.26
13.5


In [5]:
ucla_players = ['Gianluca Ballotta',
                'Kaylan Bigun',
                'Cassius Chinlund',
                'Andrei Crabel',
                'Alexander Hoogmartens',
                'Spencer Johnson',
                'Rudy Quan',
                'Giacomo Revelli',
                'Aadarsh Tripathi',
                'Emon van Loben Sels',
                'Leo von Bismarck',
                
                'Olivia Center',
                'Kate Fakih',
                'Bianca Fernandez',
                'Ahmani Guichard',
                'Kimberly Hance',
                'Mia Jovic',
                'Anne-Christine Lutkemeyer',
                'Elise Wagle']

In [6]:
# get UTRs of all ucla players and store in dictionary
ucla_utr_dict = {}
for player in ucla_players:
    print(player)
    utr = get_utr(player)
    ucla_utr_dict[player] = utr

Gianluca Ballotta
Kaylan Bigun
Cassius Chinlund
Andrei Crabel
Alexander Hoogmartens
Spencer Johnson
Rudy Quan
Giacomo Revelli
Aadarsh Tripathi
Emon van Loben Sels
Leo von Bismarck
Olivia Center
Kate Fakih
Bianca Fernandez
Ahmani Guichard
Kimberly Hance
Mia Jovic
Anne-Christine Lutkemeyer
Elise Wagle


In [7]:
ucla_utr_dict

{'Gianluca Ballotta': 12.0,
 'Kaylan Bigun': 13.19,
 'Cassius Chinlund': 12.27,
 'Andrei Crabel': 11.48,
 'Alexander Hoogmartens': 13.0,
 'Spencer Johnson': 13.5,
 'Rudy Quan': 13.91,
 'Giacomo Revelli': 12.0,
 'Aadarsh Tripathi': 12.97,
 'Emon van Loben Sels': 13.31,
 'Leo von Bismarck': 11.0,
 'Olivia Center': 10.26,
 'Kate Fakih': 10.78,
 'Bianca Fernandez': 9.99,
 'Ahmani Guichard': 10.0,
 'Kimberly Hance': 10.43,
 'Mia Jovic': 8.0,
 'Anne-Christine Lutkemeyer': 11.0,
 'Elise Wagle': 10.0}

## Getting Dataframe of Past Matches

#### getting singles results

player 1 is winner

In [5]:
# Convert UTR scoring into a string
def get_score_string(score):
    sets = []
    for i in range(1, 4):
        if str(i) in score:
            loser_score = score[str(i)]['loser'] if score[str(i)]['loser'] is not None else '0' # account for 0 stored as None in the UTR data
            set_score = f"{score[str(i)]['winner']}-{loser_score}"
            if score[str(i)]['tiebreak'] != None:
                set_score += f"({score[str(i)]['tiebreak']})"
            sets.append(set_score)
    return ', '.join(sets)

In [6]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [9]:
# Convert date into a more readable format
def convert_date(date):
    utc = datetime.fromisoformat(date).replace(tzinfo=ZoneInfo('UTC'))
    pst = utc.astimezone(ZoneInfo('America/Los_Angeles'))
    return pst.strftime('%m/%d/%Y')

In [ ]:
def get_singles_results(name):
    user_id = get_user_id(name)

    # add ?type=singles for singles only
    url = f'https://app.universaltennis.com/api/v1/player/{user_id}/results?type=singles'
    page = requests.get(url)
    data = page.json()

    results = []
    for event in data['events']:
        event_name = event['name']

        if len(event['draws']) == 0:
            for result in event['results']:
                date = convert_date(result['date'])
                player1 = result['players']['winner1']['firstName'] + ' ' + result['players']['winner1']['lastName']
                player2 = result['players']['loser1']['firstName'] + ' ' + result['players']['loser1']['lastName']
                player1_singles_utr = result['players']['winner1']['singlesUtr']
                player2_singles_utr = result['players']['loser1']['singlesUtr']
                score = get_score_string(result['score'])
                results.append([event_name, date, player1, player1_singles_utr, player2, player2_singles_utr, score])
        else:
            for result in event['draws'][0]['results']:
                date = result['date']
                player1 = result['players']['winner1']['firstName'] + ' ' + result['players']['winner1']['lastName']
                player2 = result['players']['loser1']['firstName'] + ' ' + result['players']['loser1']['lastName']
                player1_singles_utr = result['players']['winner1']['singlesUtr']
                player2_singles_utr = result['players']['loser1']['singlesUtr']
                score = get_score_string(result['score'])
                results.append([event_name, date, player1, player1_singles_utr, player2, player2_singles_utr, score])

    singles_results = pd.DataFrame(results, columns=['Event Name', 'Date', 'Player1', 'Player1 UTR', 'Player2', 'Player2 UTR', 'Score'])
    return singles_results

In [10]:
spencer_singles = get_singles_results('Spencer Johnson')
spencer_singles

,Event Name,Date,Player1,Player1 UTR,Player2,Player2 UTR,Score
0,Dual Match: Stanford University vs University ...,2025-02-02T00:00:00,Samir Banerjee,13.71,Spencer Johnson,13.5,"6-3, 3-6, 6-4"
1,"Dual Match: University of California, Berkeley...",2025-02-01T00:00:00,Spencer Johnson,13.50,Alexander Chang,13.0,"7-6(1), 6-7(7), 2-2"
2,Dual Match: University of Georgia vs Universit...,2025-01-26T00:00:00,Spencer Johnson,13.50,Thomas Paulsell,13.0,"6-2, 7-6(3)"
3,"Dual Match: University of California, Los Ange...",2025-01-25T00:00:00,Theo Papamalamis,13.57,Spencer Johnson,13.5,"6-3, 6-1"
4,Dual Match: University of San Diego vs Univers...,2025-01-20T00:00:00,Oliver Tarvet,14.00,Spencer Johnson,13.5,"7-6(2), 6-4"
...,...,...,...,...,...,...,...
583,None,2013-11-30T00:00:00,Spencer Johnson,13.50,Jensen Rideout,0.0,"6-2, 6-4"
584,None,2013-11-30T00:00:00,Spencer Johnson,13.50,Ethan Dubil,0.0,"6-2, 6-2"
585,None,2013-11-30T00:00:00,Spencer Johnson,13.50,Aidan Cheney,0.0,"6-0, 6-1"
586,None,2013-10-20T00:00:00,Dylan Harvala,0.00,Spencer Johnson,13.5,"6-0, 6-3"


In [11]:
novak_singles = get_singles_results('Novak Djokovic')
novak_singles

,Event Name,Date,Player1,Player1 UTR,Player2,Player2 UTR,Score
0,"Indian Wells, USA",2025-03-08T22:30:00,Botic VAN DE ZANDSCHULP,15.24,Novak Djokovic,16.21,"6-2, 3-6, 6-1"
1,"ATP Doha, Qatar",2025-02-18T16:20:00,Matteo Berrettini,15.82,Novak Djokovic,16.21,"7-6(4), 6-2"
2,Australian Open,2025-01-24T03:45:00,Alexander ZVEREV,15.98,Novak Djokovic,16.21,7-6(5)
3,Australian Open,2025-01-21T10:15:00,Novak Djokovic,16.21,Carlos Alcaraz,16.14,"4-6, 6-4, 6-3"
4,Australian Open,2025-01-19T08:15:00,Novak Djokovic,16.21,Jiri Lehecka,15.80,"6-3, 6-4, 7-6(4)"
...,...,...,...,...,...,...,...
931,None,2010-03-09T00:00:00,Novak Djokovic,16.21,Philipp Kohlschreiber,0.00,"6-3, 2-6, 7-6(3)"
932,None,2001-05-17T00:00:00,Novak Djokovic,16.21,Roger Federer,0.00,"6-2, 7-6(4)"
933,None,2001-05-16T00:00:00,Novak Djokovic,16.21,Juan MONACO,0.00,"4-6, 6-2, 6-3"
934,None,2001-05-16T00:00:00,Novak Djokovic,16.21,Jo-Wilfried Tsonga,0.00,"7-5, 6-1"


In [12]:
get_user_id('Spencer Johnson')

'2580878'

In [13]:
# Spencer Johnson
url = f'https://app.universaltennis.com/api/v1/player/2580878/results'
page = requests.get(url)
spencer = page.json()

In [14]:
spencer

{'wins': 566,
 'losses': 256,
 'withdrawls': 3,
 'events': [{'id': 2509783,
   'name': 'Dual Match: Stanford University vs University of California, Los Angeles',
   'isTms': False,
   'isDualMatch': True,
   'timeZone': None,
   'startDate': '2025-02-02T00:00:00',
   'endDate': '2025-02-02T00:00:00',
   'sportTypeId': None,
   'draws': [{'id': 17576478,
     'tmsId': None,
     'name': '#1 Singles',
     'teamType': 'Singles',
     'gender': 'Male',
     'results': [{'eventViewModel': None,
       'date': '2025-02-02T00:00:00',
       'id': 51131985,
       'clubIds': [],
       'players': {'winner1': {'eventViewModel': None,
         'id': '122837',
         'firstName': 'Samir',
         'lastName': 'Banerjee',
         'gender': 'M',
         'city': None,
         'state': None,
         'hasYobOnly': False,
         'singlesUtr': 13.71,
         'singlesUtrDisplay': '13.71',
         'ratingStatusSingles': 'Rated',
         'ratingProgressSingles': '100',
         'ratingStatusIm

In [15]:
spencer['events'][0]

{'id': 2509783,
 'name': 'Dual Match: Stanford University vs University of California, Los Angeles',
 'isTms': False,
 'isDualMatch': True,
 'timeZone': None,
 'startDate': '2025-02-02T00:00:00',
 'endDate': '2025-02-02T00:00:00',
 'sportTypeId': None,
 'draws': [{'id': 17576478,
   'tmsId': None,
   'name': '#1 Singles',
   'teamType': 'Singles',
   'gender': 'Male',
   'results': [{'eventViewModel': None,
     'date': '2025-02-02T00:00:00',
     'id': 51131985,
     'clubIds': [],
     'players': {'winner1': {'eventViewModel': None,
       'id': '122837',
       'firstName': 'Samir',
       'lastName': 'Banerjee',
       'gender': 'M',
       'city': None,
       'state': None,
       'hasYobOnly': False,
       'singlesUtr': 13.71,
       'singlesUtrDisplay': '13.71',
       'ratingStatusSingles': 'Rated',
       'ratingProgressSingles': '100',
       'ratingStatusImgageSingles': 'https://universaltennis.com/azure/images/icons/status-rated.png?h=17',
       'doublesUtr': 12.99,
    